# ODEs: Planetary motion (PyTree version)

Here, we simulate planetary motion for the simple 2-body Kepler case, a planar 3-body case, and the restricted 3-body problem in a co-rotating (synodic) reference frame.

This is a rewrite of `planetary.ipynb` that uses **PyTrees** (see `pytree.ipynb`) to describe the state of the system.
Instead of packing everything into a flat list like `[x1, y1, vx1, vy1, x2, y2, ...]` and keeping track of which index means what,
we write the state as a nested structure of named 2D vectors, e.g.

```python
state = {
    "earth": {"r": vector.obj(x=1.0, y=0.0), "v": vector.obj(x=0.0, y=6.28)},
    "sun":   {"r": vector.obj(x=0.0, y=0.0), "v": vector.obj(x=0.0, y=0.0)},
}
```

`scipy.integrate.solve_ivp` still only understands 1D arrays, so we wrap it: the PyTree machinery flattens the state into a 1D array
for the integrator, and unflattens it again whenever our derivative function is called.

In [ ]:
import numpy as np
import matplotlib
matplotlib.rcParams['legend.fancybox'] = True
import matplotlib.pyplot as plt
import scipy.integrate
import scipy.optimize
import vector

pytree = vector.register_pytree()

## Wrapping `solve_ivp` for PyTree states

This is the same idea as `wrapped_solve` in `pytree.ipynb`:

* `pytree.flatten(y0)` turns the nested initial state into a flat list of numbers, plus a `treedef` that remembers the structure.
* Inside `flat_fun`, we `unflatten` the flat array into the nested structure, call our human-friendly derivative function,
  and flatten the result again so `solve_ivp` gets the 1D array it expects.
* At the end, we unflatten the full solution `res.y` (shape `(n_variables, n_times)`). Each leaf then becomes an array over time,
  so e.g. `solution["earth"]["r"].x` is the $x$ trajectory of the Earth.

The derivative function must return a PyTree with **the same structure** as the state: for each `"r"` it returns $d\vec r/dt$, and for each `"v"` it returns $d\vec v/dt$.

In [ ]:
def solve_ivp_pytree(fun, t_span, y0, **kwargs):
    '''
    Wrapper around scipy.integrate.solve_ivp for states that are PyTrees.

    fun(t, state) -> dstate_dt, where state and dstate_dt have the same PyTree structure.
    Returns (t, solution, res): solution has the same structure as y0, with each leaf
    an array over the output times; res is the raw solve_ivp result.
    '''
    flat_y0, treedef = pytree.flatten(y0)

    def flat_fun(t, flat_y):
        state = pytree.unflatten(treedef, flat_y)
        dstate_dt = fun(t, state)
        flat_dstate_dt, _ = pytree.flatten(dstate_dt)
        return np.asarray(flat_dstate_dt)

    res = scipy.integrate.solve_ivp(fun=flat_fun, t_span=t_span, y0=np.asarray(flat_y0), **kwargs)
    solution = pytree.unflatten(treedef, res.y)
    return res.t, solution, res

## Kepler motion

The state is a dictionary with a position vector `"r"` and velocity vector `"v"`. The equations of motion read just like the physics:

$$\frac{d\vec r}{dt} = \vec v, \qquad \frac{d\vec v}{dt} = -\frac{G(m_1+m_2)}{r^3}\,\vec r .$$

The optional `step_using_y` flag changes the independent variable from $t$ to $y$ by dividing every component of the flow by $v_y$.
`pytree.map` applies a function to every leaf of the tree, so this is a one-liner.

In [ ]:
class Kepler:
    '''
    Functor implementing 2-body problem in 2 dimensions.
    State: {"r": Vector2D, "v": Vector2D}
    '''
    def __init__(
            self,
            G_m1_plus_m2: float,
            r_aphelion: float,
            eccentricity: float,
            step_using_y: bool = False):

        self._G_m1_plus_m2 = G_m1_plus_m2
        self._r_aphelion = r_aphelion
        self._eccentricity = eccentricity
        self._a = self._r_aphelion / (1 + self._eccentricity)
        self._vy0 = np.sqrt(self._G_m1_plus_m2 * (2 / self._r_aphelion - 1 / self._a))
        self._step_using_y = step_using_y

    def r_aphelion(self):
        return self._r_aphelion

    def vy0(self):
        return self._vy0

    def initial_state(self):
        return {
            "r": vector.obj(x=self._r_aphelion, y=0.0),
            "v": vector.obj(x=0.0, y=self._vy0),
        }

    def __call__(self, t, state):
        r, v = state["r"], state["v"]
        flow = {
            "r": v,                                         # dr/dt
            "v": -self._G_m1_plus_m2 * r / r.rho**3,        # dv/dt
        }
        if self._step_using_y:            # change independent variable from t to y
            flow = pytree.map(lambda leaf: leaf / v.y, flow)
        return flow

In [ ]:
kepler = Kepler(G_m1_plus_m2 = 1.0, r_aphelion = 1.0, eccentricity = 0.0)

t, sol, res = solve_ivp_pytree(kepler,
                               t_span=(0, 200.),
                               y0=kepler.initial_state(),
                               atol=1e-8, rtol=1e-6,
                               method="RK45",
                               first_step=0.001)
r = sol["r"]
v = sol["v"]

In [ ]:
fig = plt.figure(1, figsize=(6,6))
plt.scatter(r.x, r.y)
plt.xlabel('x (AU)')
plt.ylabel('y (AU)')
plt.grid()

Because the solution comes back as vectors, derived quantities are easy to compute. For example, the specific energy
$E = \frac{1}{2}v^2 - G(m_1+m_2)/r$ should be conserved; any drift tells us about the integration error.

In [ ]:
energy = 0.5 * v.rho**2 - 1.0 / r.rho
plt.figure(figsize=(6,3))
plt.plot(t, energy - energy[0])
plt.xlabel('t')
plt.ylabel(r'$E(t) - E(0)$')
plt.grid()

## Planar 3-body motion

Now the state is a dictionary of bodies, each with its own `"r"` and `"v"`:

```python
state = {"moon": {"r": ..., "v": ...}, "earth": {"r": ..., "v": ...}, "sun": {"r": ..., "v": ...}}
```

Since every body is labeled, we no longer need to write out each of the nine pairwise force terms by hand: we can just loop over
pairs of bodies,

$$\frac{d\vec v_i}{dt} = -\sum_{j\neq i} G m_j \frac{\vec r_i - \vec r_j}{|\vec r_i - \vec r_j|^3}.$$

Nothing in the functor below is specific to 3 bodies, so it works for any number of bodies.

In [ ]:
class PlanarNBody:
    '''
    Functor implementing the N-body problem in 2 dimensions.
    State: {name: {"r": Vector2D, "v": Vector2D}} for each body.
    '''

    def __init__(self, masses: dict):
        self._masses = masses
        # use units such that G*(total mass) = 4*pi**2
        self._G = 4 * np.pi**2 / sum(masses.values())

    def __call__(self, t, state):
        flow = {}
        for i, body_i in state.items():
            a_i = vector.obj(x=0.0, y=0.0)
            for j, body_j in state.items():
                if i == j:
                    continue
                dr = body_i["r"] - body_j["r"]
                a_i = a_i - self._G * self._masses[j] * dr / dr.rho**3
            flow[i] = {"r": body_i["v"], "v": a_i}
        return flow

In [ ]:
masses = {"moon": 0.012, "earth": 1.0, "sun": 330000.0}
planar3body = PlanarNBody(masses)

y0 = {
    "moon":  {"r": vector.obj(x=0.0, y=1.0027), "v": vector.obj(x=6.724, y=0.0)},
    "earth": {"r": vector.obj(x=0.0, y=1.0),    "v": vector.obj(x=6.5,   y=0.0)},
    "sun":   {"r": vector.obj(x=0.0, y=0.0),    "v": vector.obj(x=0.0,   y=0.0)},
}

t, sol, res = solve_ivp_pytree(planar3body,
                               t_span=(0, 1.27),   # years
                               y0=y0,
                               atol=1e-8, rtol=1e-6,
                               method="RK45",
                               first_step=0.001)

In [ ]:
fig = plt.figure(2, figsize=(6,6))
plt.scatter(sol["moon"]["r"].x,  sol["moon"]["r"].y,  facecolors='none', edgecolors='r', marker="o", s=1,  label="moon")
plt.scatter(sol["earth"]["r"].x, sol["earth"]["r"].y, facecolors='none', edgecolors='b', marker="o", s=10, label="earth")
plt.scatter(sol["sun"]["r"].x,   sol["sun"]["r"].y,   facecolors='none', edgecolors='k', marker="o", s=20, label="sun")
plt.xlabel('x (AU)')
plt.ylabel('y (AU)')
plt.legend()
plt.show()

The Moon's orbit is much easier to see relative to the Earth. With named vectors, that's just a subtraction:

In [ ]:
r_moon_earth = sol["moon"]["r"] - sol["earth"]["r"]
fig = plt.figure(figsize=(6,6))
plt.plot(r_moon_earth.x, r_moon_earth.y, 'r-')
plt.xlabel(r'$x_{moon} - x_{earth}$ (AU)')
plt.ylabel(r'$y_{moon} - y_{earth}$ (AU)')
plt.gca().set_aspect('equal')
plt.grid()

## Restricted 3-body motion

In the co-rotating frame (with the units chosen in the lecture notes, so the only parameter is $\alpha = m_2/(m_1+m_2)$), the test mass feels
gravity from the two primaries at $\vec r_1 = (\alpha - 1, 0)$ and $\vec r_2 = (\alpha, 0)$, plus the centrifugal and Coriolis forces:

$$\frac{d\vec v}{dt} = -(1-\alpha)\frac{\vec r - \vec r_2}{|\vec r - \vec r_2|^3} - \alpha\frac{\vec r - \vec r_1}{|\vec r - \vec r_1|^3}
+ \vec r + 2\,(v_y, -v_x).$$

The last two terms are the centrifugal force $-\hat z\times(\hat z\times\vec r) = \vec r$ and the Coriolis force $-2\hat z\times\vec v = 2(v_y, -v_x)$ (with $\Omega = 1$).

In [ ]:
class Rcp3Body:
    '''
    Functor implementing restricted 3-body planar motion.
    Following the presentation in the lecture notes, we
      have only one constant, alpha, via a particular
      choice of units.
    State: {"r": Vector2D, "v": Vector2D}
    '''

    def __init__(self, alpha):
        # alpha = m2/(m1+m2) in the webnotes
        self._alpha = alpha
        self._eps = 1e-6

        # positions of the two primaries
        self._r1 = vector.obj(x=self._alpha - 1, y=0.0)
        self._r2 = vector.obj(x=self._alpha,     y=0.0)

        # Lagrange points
        self._L = np.array( [
            [scipy.optimize.bisect(self.f_x, self._alpha - 1 + self._eps, self._alpha - self._eps), 0.0],
            [scipy.optimize.bisect(self.f_x, -1.5, self._alpha - 1 - self._eps), 0.0],
            [scipy.optimize.bisect(self.f_x, self._alpha + self._eps, 1.5 ), 0.0],
            [self._alpha - 0.5,  np.sqrt(3.0) / 2 ], [self._alpha - 0.5, -np.sqrt(3.0) / 2 ]
        ]
        )

    def alpha(self):
        return self._alpha

    def f_x(self, x):
        '''
        effective x component of force on the x-axis
        '''
        return (x - (1 - self._alpha) * (x - self._alpha) / np.abs(x - self._alpha) / (x - self._alpha)**2
                   - self._alpha * (x + 1 - self._alpha) / np.abs(x + 1 - self._alpha) / (x + 1 - self._alpha)**2)

    def __call__(self, t, state):
        '''
        equations in co-rotating frame
        '''
        r, v = state["r"], state["v"]
        d2 = r - self._r2          # displacement from mass (1 - alpha)
        d1 = r - self._r1          # displacement from mass alpha
        gravity     = -(1 - self._alpha) * d2 / d2.rho**3 - self._alpha * d1 / d1.rho**3
        centrifugal = r
        coriolis    = 2 * vector.obj(x=v.y, y=-v.x)
        return {"r": v, "v": gravity + centrifugal + coriolis}

In [ ]:
rcp3body = Rcp3Body(alpha=0.25)

y0 = {"r": vector.obj(x=-0.35, y=0.0), "v": vector.obj(x=0.0, y=0.5)}

t, sol, res = solve_ivp_pytree(rcp3body,
                               t_span=(0., 10.),
                               y0=y0,
                               atol=1e-8, rtol=1e-6,
                               method="RK45",
                               first_step=0.001)
r = sol["r"]
v = sol["v"]

In [ ]:
fig = plt.figure(3, figsize=(6,6))
plt.scatter(rcp3body._L[:,0], rcp3body._L[:,1], color='b', s=10)
plt.scatter(r.x, r.y, color='r', s=1)
plt.scatter([rcp3body.alpha()-1], [0], color='g', s=100*(1-rcp3body.alpha()))
plt.scatter([rcp3body.alpha()],   [0], color='g', s=100*rcp3body.alpha())
plt.xlim(-2,2)
plt.ylim(-2,2)
plt.annotate('r1', xy=(rcp3body.alpha()-1, 0), xycoords='data',
             xytext=(rcp3body.alpha()-1, 0.1), textcoords='data')
plt.annotate('r2', xy=(rcp3body.alpha(), 0), xycoords='data',
             xytext=(rcp3body.alpha(), 0.1), textcoords='data')
for i, L in enumerate(rcp3body._L):
    plt.annotate(f'L{i+1}', xy=L, xycoords='data',
                 xytext=L + np.array([0.0, 0.1]), textcoords='data')
plt.show()

## Comparing the two approaches

| | flat-list version (`planetary.ipynb`) | PyTree version (this notebook) |
|---|---|---|
| state | `[x1, y1, vx1, vy1, x2, ...]` | `{"moon": {"r": ..., "v": ...}, ...}` |
| accessing a variable | `states[8]` (which body was that?) | `sol["sun"]["r"].x` |
| forces | written out component by component | vector expressions: `-G*m*dr/dr.rho**3` |
| adding a body | renumber every index | add an entry to `masses` and `y0` |

The cost is a small amount of overhead: every call to the derivative function flattens and unflattens the state,
which is slower than indexing a NumPy array directly. For large simulations you would want to vectorize over bodies
(e.g. with `vector.array` and NumPy broadcasting) instead of looping over pairs in Python.